## Baseline Erf Model

In [1]:
# @title
!pip install lightning > /dev/null
!pip install gitignore_parser > /dev/null
!pip install -U 'jsonargparse[signatures]>=4.27.7' >/dev/null

In [2]:
# @title
from google.colab import drive
import os
import sys
import json
import yaml
import torch
import importlib
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm import tqdm
from lightning import seed_everything

# 1. Mount Drive and Configure Paths
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive')

project_root = '/content/drive/MyDrive/FundGitHubProject'
if not os.path.exists('/content/ProjectFolder'):
  # creates shortcut to access the project folder
  !ln -s /content/drive/MyDrive/FundGitHubProject /content/ProjectFolder
eomt_folder = project_root + '/eomt'

os.chdir(project_root)
if project_root not in sys.path:
    sys.path.insert(0, project_root)
if eomt_folder not in sys.path:
    sys.path.insert(0, eomt_folder)

from eval.iouEval import iouEval
seed_everything(0, verbose=False)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Active Device: {device}")

Mounted at /content/drive
Active Device: cuda


In [3]:
import torch

erfnet_pretrained = torch.load('trained_models/erfnet_pretrained.pth')

In [ ]:
# paper self contained
#
import torch
import os
from eomt.datasets.cityscapes_semantic import CityscapesSemantic

# Import the model class (usually named Net or ERFNet in standard implementations)
try:
    from eval.erfnet import Net as ERFNet
except ImportError:
    from eval.erfnet import ERFNet

# Provide the path to the Cityscapes dataset
dataset_path = os.path.join(project_root, 'eomt/data') # adjust this path as needed

# Instantiate the DataModule (it needs self)
cityscapes_dm = CityscapesSemantic(path=dataset_path)

# Lightning DataModules typically require setup before getting dataloaders
cityscapes_dm.setup(stage='validate')
val_dataloader = cityscapes_dm.val_dataloader()

# Remove 'module.' prefix from state dict keys (caused by DataParallel)
cleaned_state_dict = {k.replace('module.', ''): v for k, v in erfnet_pretrained.items()}

# Initialize the model (Cityscapes typically has 19 or 20 classes)
try:
    model = ERFNet(19)
    model.load_state_dict(cleaned_state_dict)
except RuntimeError:
    # Fallback to 20 classes if state_dict shapes mismatch
    model = ERFNet(20)
    model.load_state_dict(cleaned_state_dict)

# Move model to device and set to evaluation mode
model = model.to(device)
model.eval()

for batch_idx, batch in enumerate(val_dataloader):
    # Lightning dataloaders usually return (images, targets)
    images, targets = batch
    images = images.to(device)

    # Forward pass without calculating gradients
    with torch.no_grad():
        outputs = model(images)

    print(f"Successfully processed batch {batch_idx}! Output shape: {outputs.shape}")
    break


In [ ]:
from